In [ ]:
import hockey_scraper 

# This script scrapes NHL data for the 2025 season playoffs and saves it to a CSV file.
hockey_scraper.scrape_season('2025', 'playoffs', True)


In [ ]:
from pynhl import Scoreboard

scores = Scoreboard()
games_today = scores.games()

for game in games_today:
    print(game['homeTeam']['teamName'], 'vs', game['awayTeam']['teamName'])

Now = 7 days

In [ ]:
import requests
import pandas as pd

url = "https://api-web.nhle.com/v1/scoreboard/now"
res = requests.get(url)
data = res.json()

games = []

for day in data['gamesByDate']:
    for game in day['games']:
        games.append({
            'date': day['date'],
            'home': game['homeTeam']['abbrev'],
            'away': game['awayTeam']['abbrev'],
            'startTime': game['startTimeUTC'],
            'gameState': game['gameState']
        })

df = pd.DataFrame(games)
print(df)


In [ ]:
import requests
import pandas as pd

# Use today or any playoff date
url = "https://api-web.nhle.com/v1/schedule/now"
headers = {'User-Agent': 'Mozilla/5.0'}

res = requests.get(url, headers=headers)

if res.status_code != 200:
    raise Exception(f"Request failed: {res.status_code}")

data = res.json()

all_playoff_games = []

for day in data.get('gameWeek', []):
    date = day['date']
    for game in day.get('games', []):
        if game.get('gameType') == 3:  # Playoff game
            all_playoff_games.append({
                'date': date,
                'homeTeam': game['homeTeam']['abbrev'],
                'awayTeam': game['awayTeam']['abbrev'],
                'startTimeUTC': game['startTimeUTC'],
                'gameState': game['gameState'],
                'homeScore': game['homeTeam'].get('score'),
                'awayScore': game['awayTeam'].get('score'),
                'seriesStatus': game.get('seriesStatus', {}).get('seriesTitle', '') + " - Game " + str(game.get('seriesStatus', {}).get('gameNumberOfSeries', ''))
            })

# Build DataFrame
df = pd.DataFrame(all_playoff_games)

print(f"✅ Fetched {len(df)} playoff games")
print(df)

# Optional: Save
# df.to_csv("playoff_games_2025.csv", index=False)


In [ ]:
import requests
import pandas as pd
from datetime import datetime

# Start date and end date
start_date = datetime(2025, 4, 19)
end_date = datetime(2025, 4, 28)

# Request only ONCE
url = f"https://api-web.nhle.com/v1/schedule/{start_date.strftime('%Y-%m-%d')}"
res = requests.get(url)

all_games = []

if res.status_code == 200:
    data = res.json()

    for day in data.get('gameWeek', []):
        for game in day.get('games', []):
            if game.get('gameType') == 3:  # Only playoff games
                # Only add games within the date range we want
                game_date = datetime.strptime(day['date'], "%Y-%m-%d")
                if start_date <= game_date <= end_date:
                    all_games.append({
                        'date': day['date'],
                        'homeTeam': game['homeTeam']['abbrev'],
                        'awayTeam': game['awayTeam']['abbrev'],
                        'startTimeUTC': game['startTimeUTC'],
                        'gameState': game['gameState'],
                        'homeScore': game['homeTeam'].get('score'),
                        'awayScore': game['awayTeam'].get('score'),
                        'seriesStatus': game.get('seriesStatus', {}).get('seriesTitle', '') + " - Game " + str(game.get('seriesStatus', {}).get('gameNumberOfSeries', ''))
                    })
else:
    print(f"❌ Failed to fetch games (status {res.status_code})")

# Done collecting
df = pd.DataFrame(all_games)
print(f"✅ Fetched {len(df)} playoff games from {start_date.date()} to {end_date.date()}")
display(df)

# Optional save
# df.to_csv("playoff_games_filtered.csv", index=False)


In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# Start from beginning of playoffs
start_date = datetime(2025, 4, 19)
# End at known playoff end (example June 24)
end_date = datetime(2025, 6, 24)

delta = timedelta(days=7)  # move by 7 days (week by week)

all_games = []

current_date = start_date

while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    url = f"https://api-web.nhle.com/v1/schedule/{date_str}"
    res = requests.get(url)

    if res.status_code == 200:
        data = res.json()

        for day in data.get('gameWeek', []):
            for game in day.get('games', []):
                if game.get('gameType') == 3:  # Playoffs
                    game_date = datetime.strptime(day['date'], "%Y-%m-%d")
                    if start_date <= game_date <= end_date:
                        all_games.append({
                            'date': day['date'],
                            'homeTeam': game['homeTeam']['abbrev'],
                            'awayTeam': game['awayTeam']['abbrev'],
                            'startTimeUTC': game['startTimeUTC'],
                            'gameState': game['gameState'],
                            'homeScore': game['homeTeam'].get('score'),
                            'awayScore': game['awayTeam'].get('score'),
                            'seriesStatus': game.get('seriesStatus', {}).get('seriesTitle', '') + " - Game " + str(game.get('seriesStatus', {}).get('gameNumberOfSeries', ''))
                        })
    else:
        print(f"⚠️ Failed fetching week starting {date_str}")

    current_date += delta  # move one week ahead

# Done
df = pd.DataFrame(all_games)
print(f"✅ Fetched {len(df)} playoff games from {start_date.date()} to {end_date.date()}")
display(df)

# Optional save
# df.to_csv("all_playoff_games_2025.csv", index=False)
